[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tarekFerdous/python-practice/blob/main/exercises/0001-hash-maps/exercises.ipynb)

# Lesson 1 — Hash maps (dict, Counter, defaultdict) + sorting with a key

Lesson: [0001-hash-maps.html](https://github.com/tarekFerdous/python-practice/blob/main/lessons/0001-hash-maps.html) (open the HTML locally for the quizzes)

**How to use:** run the setup cell once. Then for each exercise: edit the function cell, run it (`Shift+Enter`), then run the check cell under it. Each check prints PASS/FAIL with what you returned vs what was expected. `Run All` checks everything.

Before coding each one, say out loud: brute force + its Big-O, then your idea. When it passes, give **time and space** complexity.

In [ ]:
# Setup: run this cell first (test helper, same as assets/nbcheck.py)
"""Tiny test runner for the exercise notebooks.

    check(two_sum, [
        (([2, 7, 11, 15], 9), (0, 1)),   # (args tuple, expected)
    ])

Prints one line per case: what you called, what you got, what was expected.
Exceptions are caught and shown instead of stopping the notebook.
"""
import copy

GREEN, RED, RESET = "\033[32m", "\033[31m", "\033[0m"


def check(fn, cases, name=None):
    name = name or getattr(fn, "__qualname__", repr(fn))
    passed = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:  # show the crash, keep going
            got = f"<{type(e).__name__}: {e}>"
        ok = got == expected and type(got) is type(expected)
        passed += ok
        mark = f"{GREEN}PASS{RESET}" if ok else f"{RED}FAIL{RESET}"
        line = f"{mark}  {name}({shown}) -> {got!r}"
        if not ok:
            line += f"   expected {expected!r}"
        print(line)
    colour = GREEN if passed == len(cases) else RED
    print(f"{colour}{passed}/{len(cases)} passed{RESET}\n")
    return passed == len(cases)

## 1. `two_sum`

In [ ]:
def two_sum(nums: list[int], target: int) -> tuple[int, int]:
    """Return the indices (i, j), i < j, of the two numbers that add up to target.
    Exactly one answer exists. Must be O(n), not the O(n^2) double loop.

    two_sum([2, 7, 11, 15], 9) -> (0, 1)
    two_sum([3, 2, 4], 6)      -> (1, 2)
    """
    visited = {}
    for i, v in enumerate(nums):
        remain = target - v
        if remain in visited:
            return (visited[remain], i)
        visited[v] = i

In [ ]:
check(two_sum, [
    (([2, 7, 11, 15], 9), (0, 1)),
    (([3, 2, 4], 6), (1, 2)),
    (([3, 3], 6), (0, 1)),
    (([-1, 5, 10, -4], 6), (2, 3)),
])

## 2. `top_k_words`

In [ ]:
def top_k_words(text: str, k: int) -> list[str]:
    """Return the k most frequent words, lowercase.
    Ties: alphabetical order. Words are separated by whitespace; strip . , ! ? from them.

    top_k_words("the cat and the hat. The end!", 2) -> ["the", "and"]
    """
    text = text.replace('.', '').replace(',', '').replace('!', '').replace('?', '').lower()
    text = text.split(' ')
    print(text)
    words = {}
    for i, word in enumerate(text):
        if word in words:
            words[word] +=1
        else:
            words[word] = 1
    
    return list(sorted(words, key=lambda w: (-words[w], w)))[0:k]

In [ ]:
check(top_k_words, [
    (("the cat and the hat. The end!", 2), ["the", "and"]),
    (("b a c b a", 3), ["a", "b", "c"]),
    (("Hi! hi? HI.", 1), ["hi"]),
])

## 3. `group_anagrams`

In [ ]:
def group_anagrams(words: list[str]) -> list[list[str]]:
    """Group words that are anagrams of each other.
    Keep each group in input order; order the groups by first appearance.

    group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
      -> [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]
    """
    storage = {}
    for i, v in enumerate(words):
        word = ''.join(sorted(list(v)))
        if word in storage:
            storage[word].append(v)
        else:
            storage[word] = [v]
    return [storage[v] for v in storage]

In [ ]:
check(group_anagrams, [
    ((["eat", "tea", "tan", "ate", "nat", "bat"],), [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]),
    (([],), []),
    ((["a"],), [["a"]]),
])

---
## Teacher's idiomatic versions
Read after yours pass, compare, then rewrite from memory.

In [ ]:
from collections import Counter, defaultdict
import heapq

In [ ]:
def two_sum_v2(nums: list[int], target: int) -> tuple[int, int]:
    # O(n) time, O(n) space
    seen = {}                                   # value -> index
    for i, num in enumerate(nums):
        if target - num in seen:
            return (seen[target - num], i)      # earlier index is always smaller
        seen[num] = i

In [ ]:
check(two_sum_v2, [
    (([2, 7, 11, 15], 9), (0, 1)),
    (([3, 2, 4], 6), (1, 2)),
    (([3, 3], 6), (0, 1)),
    (([-1, 5, 10, -4], 6), (2, 3)),
])

In [ ]:
def top_k_words_v2(text: str, k: int) -> list[str]:
    # O(n + m log m) time, O(m) space  (n words, m distinct)
    words = [w.strip(".,!?").lower() for w in text.split()]
    counts = Counter(words)
    return sorted(counts, key=lambda w: (-counts[w], w))[:k]

In [ ]:
check(top_k_words_v2, [
    (("the cat and the hat. The end!", 2), ["the", "and"]),
    (("b a c b a", 3), ["a", "b", "c"]),
    (("Hi! hi? HI.", 1), ["hi"]),
])

In [ ]:
def top_k_words_heap(text: str, k: int) -> list[str]:
    # O(n + m log k) time: better when k is much smaller than m
    counts = Counter(w.strip(".,!?").lower() for w in text.split())
    return heapq.nsmallest(k, counts, key=lambda w: (-counts[w], w))

In [ ]:
check(top_k_words_heap, [
    (("the cat and the hat. The end!", 2), ["the", "and"]),
    (("b a c b a", 3), ["a", "b", "c"]),
    (("Hi! hi? HI.", 1), ["hi"]),
])

In [ ]:
def group_anagrams_v2(words: list[str]) -> list[list[str]]:
    # O(n * L log L) time, O(n * L) space  (L = word length)
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    return list(groups.values())

In [ ]:
check(group_anagrams_v2, [
    ((["eat", "tea", "tan", "ate", "nat", "bat"],), [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]),
    (([],), []),
    ((["a"],), [["a"]]),
])

In [ ]:
def group_anagrams_count_key(words: list[str]) -> list[list[str]]:
    # O(n * L) time: letter counts as the key instead of sorting each word
    groups = defaultdict(list)
    for w in words:
        key = [0] * 26
        for ch in w:
            key[ord(ch) - ord("a")] += 1
        groups[tuple(key)].append(w)            # lists can't be dict keys; tuples can
    return list(groups.values())

In [ ]:
check(group_anagrams_count_key, [
    ((["eat", "tea", "tan", "ate", "nat", "bat"],), [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]),
    (([],), []),
    ((["a"],), [["a"]]),
])